# TRAIN on DDR (paper 2 pilot): same recipe as `TRAIN_eyepacs_v2_CANONICAL`, different training data

Trains 4 architectures x 3 seeds = 12 models on DDR, with **every hyperparameter, augmentation, loss, optimiser, schedule and checkpoint rule copied from the canonical EyePACS notebook**. The only change is the training data.

- **Architectures (5-class):** DenseNet121, EfficientNet-B3, ResNet50, RETFound. EfficientNet-B3 replaces ViT-Base because ViT-Base needed a separate retraining fix on EyePACS (`TRAIN_eyepacs_v2_VIT_ALLSEEDS`); keeping to the canonical recipe keeps the comparison clean.
- **Data:** DDR's own `train` file for training and `valid` file for best-epoch selection, restricted to the locked DDR cohort (grades 0 to 4, byte-duplicates removed) and the v2 preprocessing (512x512 PNGs already in `08_ddr/processed_v2`). DDR's `test` file is scored for a sanity check only.
- **Outputs:** checkpoints `02_checkpoints/ddr_<model>[_seedK]_v2.pt`; float64 predictions on DDR valid and test in `03_predictions/ddrtrained_predictions_<split>_<model>[_seedK]_v2.csv`.
- **Resumable:** any model whose checkpoint exists is skipped.

A100, High-RAM; `HF_TOKEN` secret for RETFound. Roughly 5 to 7 hours in total (RETFound is about 80% of it); safe to run across several sessions.

In [1]:
# Cell 1: environment, paths, RETFound preflight
from google.colab import drive, userdata
drive.mount('/content/drive')
!pip -q install timm transformers
import os, io, glob, json, time, zipfile, datetime, random, hashlib
import numpy as np, pandas as pd, torch, torch.nn as nn
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))

BASE = '/content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2'
SPLITS, CKPT, PREDS, AUDIT, DDR_IN = (f'{BASE}/{d}' for d in ('01_splits','02_checkpoints','03_predictions','99_audit','08_ddr'))
RETFOUND_REPO = 'iszt/RETFound_mae_meh'
NUM_WORKERS = 8
def sha256_file(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
def find_blocks(vit):
    best = None
    for _, mod in vit.named_modules():
        if isinstance(mod, nn.ModuleList) and len(mod) >= 12 and (best is None or len(mod) > len(best)): best = mod
    if best is None: raise RuntimeError('could not locate transformer blocks')
    return best
from transformers import AutoModel, AutoConfig
_cfg = AutoConfig.from_pretrained(RETFOUND_REPO)
assert _cfg.hidden_size == 1024 and _cfg.num_hidden_layers == 24, 'not ViT-Large'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE', '| RETFound config OK')

Mounted at /content/drive


config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

GPU: NVIDIA A100-SXM4-40GB | RETFound config OK


In [2]:
# Cell 2: roster (hyperparameters copied verbatim from the canonical EyePACS notebook)
MODELS = [
  {'name':'densenet121_5class','arch':'densenet121','task':'5class','nc':5,'epochs':15,'bs':32,'lr':1e-4,'wd':1e-4},
  {'name':'efficientnet_b3_5class','arch':'efficientnet_b3','task':'5class','nc':5,'epochs':15,'bs':32,'lr':1e-4,'wd':1e-4},
  {'name':'resnet50_5class','arch':'resnet50','task':'5class','nc':5,'epochs':15,'bs':32,'lr':1e-4,'wd':1e-4},
  {'name':'retfound_5class','arch':'retfound','task':'5class','nc':5,'epochs':15,'bs':8,'lr':5e-5,'wd':0.05,'clip':1.0,'lldecay':0.65},
]
SEEDS = [0, 1, 2]
def tag(k): return '' if k == 0 else f'_seed{k}'
BASE_SEED = 20261001            # distinct from the EyePACS runs; seed = BASE_SEED + 10*model_index + replicate
IMG_SIZE = 224
print([m['name'] for m in MODELS], 'x', len(SEEDS), 'seeds')

['densenet121_5class', 'efficientnet_b3_5class', 'resnet50_5class', 'retfound_5class'] x 3 seeds


In [3]:
# Cell 3: DDR training data (locked cohort, DDR's own train/valid/test files), images to local disk
man = pd.read_csv(f'{SPLITS}/ddr_manifest_v2.csv', dtype={'image_id': str})
coh = pd.read_csv(f'{SPLITS}/ddr_split_intraset_v2.csv', dtype={'image_id': str})[['image_id']]
d = coh.merge(man[['image_id','grade','source_split']], on='image_id', how='left')
assert d.grade.notna().all() and (d.grade <= 4).all()
d['level'] = d.grade.astype(int); d['binary_label'] = (d.level >= 2).astype(int)
src = d.source_split.str.lower()
splits = {'train': d[src.str.startswith('train')].reset_index(drop=True),
          'val':   d[src.str.startswith('val')].reset_index(drop=True),
          'test':  d[src.str.startswith('test')].reset_index(drop=True)}
for s, df in splits.items(): print(f'{s:5s} {len(df):5d} images | grade counts {df.level.value_counts().sort_index().to_dict()}')

IMG = '/content/ddr_png'; os.makedirs(IMG, exist_ok=True)
zips = sorted(glob.glob(f'{DDR_IN}/processed_v2/ddr_processed_batch*_v2.zip')); assert len(zips) == 4, zips
if len(glob.glob(f'{IMG}/*.png')) < len(d):
    for z in zips:
        with zipfile.ZipFile(z) as zf: zf.extractall(IMG)
INDEX = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob(f'{IMG}/*.png')}
missing = sorted(set(d.image_id) - set(INDEX)); assert not missing, missing[:10]
print(f'{len(INDEX):,} preprocessed DDR images on local disk')

train  6210 images | grade counts {0: 3083, 1: 315, 2: 2238, 3: 118, 4: 456}
val    2487 images | grade counts {0: 1237, 1: 126, 2: 895, 3: 47, 4: 182}
test   3725 images | grade counts {0: 1846, 1: 189, 2: 1344, 3: 71, 4: 275}
12,422 preprocessed DDR images on local disk


In [4]:
# Cell 4: dataset, builders, optimiser, training loop (canonical code, unchanged apart from data)
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import roc_auc_score
from PIL import Image
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]
TF_TRAIN = transforms.Compose([transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8,1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.ColorJitter(0.2,0.2,0.1), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
TF_EVAL = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
class DR(Dataset):
    def __init__(self, df, tf): self.df = df.reset_index(drop=True); self.tf = tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        return self.tf(Image.open(INDEX[str(r['image_id'])]).convert('RGB')), int(r['level']), int(r['binary_label']), i
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
class RETFound(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.base = AutoModel.from_pretrained(RETFOUND_REPO, add_pooling_layer=False)
        h = self.base.config.hidden_size
        self.fc_norm = nn.LayerNorm(h, eps=1e-6); self.classifier = nn.Linear(h, num_classes)
        nn.init.trunc_normal_(self.classifier.weight, std=0.01); nn.init.zeros_(self.classifier.bias)
    def forward(self, x):
        h = self.base(pixel_values=x).last_hidden_state
        return self.classifier(self.fc_norm(h[:, 1:].mean(dim=1)))
def build(cfg):
    if cfg['arch'] == 'retfound':
        m = RETFound(cfg['nc']); n = sum(p.numel() for p in m.base.parameters())
        if n < 2.9e8: raise RuntimeError(f'RETFound encoder only {n/1e6:.0f}M params, weights did not load')
        return m
    return timm.create_model(cfg['arch'], pretrained=True, num_classes=cfg['nc'])
def make_optimizer(model, cfg):
    if cfg['arch'] != 'retfound':
        return torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['wd'])
    d_, lr = cfg['lldecay'], cfg['lr']; blocks = find_blocks(model.base); n = len(blocks)
    block_ids = {id(p) for blk in blocks for p in blk.parameters()}
    embed = [p for nm, p in model.base.named_parameters() if id(p) not in block_ids and 'embed' in nm.lower()]
    embed_ids = {id(p) for p in embed}
    rest = [p for p in model.base.parameters() if id(p) not in block_ids and id(p) not in embed_ids]
    groups = [{'params': embed, 'lr': lr * (d_ ** n)}]
    for i, blk in enumerate(blocks): groups.append({'params': list(blk.parameters()), 'lr': lr * (d_ ** (n - 1 - i))})
    groups.append({'params': rest + list(model.fc_norm.parameters()), 'lr': lr})
    groups.append({'params': list(model.classifier.parameters()), 'lr': lr * 10})
    return torch.optim.AdamW(groups, lr=lr, weight_decay=cfg['wd'])
@torch.no_grad()
def predict(model, df, dev, bs=64):
    model.eval(); probs = np.zeros(len(df), dtype=np.float64)
    for x, _, _, idx in DataLoader(DR(df, TF_EVAL), batch_size=bs, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True):
        out = model(x.to(dev, non_blocking=True)).float()
        p = torch.sigmoid(out[:, 0]) if out.shape[1] == 1 else torch.softmax(out, 1)[:, 2:].sum(1)
        probs[idx.numpy()] = p.double().cpu().numpy()
    return probs
def train_one(cfg, dev, seed, eval_bs):
    seed_everything(seed); tr, va = splits['train'], splits['val']
    model = build(cfg).to(dev)
    g = torch.Generator(); g.manual_seed(seed)
    dl_ = DataLoader(DR(tr, TF_TRAIN), batch_size=cfg['bs'], shuffle=True, num_workers=NUM_WORKERS, drop_last=True,
                     generator=g, pin_memory=True, persistent_workers=True, prefetch_factor=4)
    opt = make_optimizer(model, cfg); sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg['epochs'])
    lossf = nn.BCEWithLogitsLoss() if cfg['nc'] == 1 else nn.CrossEntropyLoss(); scaler = torch.amp.GradScaler('cuda')
    best_auc, best_state, best_ep = -1.0, None, -1
    for ep in range(cfg['epochs']):
        model.train(); tot = 0.0; t0 = time.time()
        for x, lvl, lab, idx in dl_:
            x = x.to(dev, non_blocking=True); y = (lab.float() if cfg['nc'] == 1 else lvl.long()).to(dev, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                out = model(x); loss = lossf(out[:, 0], y) if cfg['nc'] == 1 else lossf(out, y)
            scaler.scale(loss).backward()
            if cfg.get('clip'): scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['clip'])
            scaler.step(opt); scaler.update(); tot += float(loss.item())
        sch.step()
        vp = predict(model, va, dev, eval_bs); vauc = float(roc_auc_score(va['binary_label'].values, vp)); star = ''
        if vauc > best_auc:
            best_auc, best_ep = vauc, ep + 1; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}; star = '  <- best'
        print(f"  epoch {ep+1}/{cfg['epochs']}  loss {tot/max(1,len(dl_)):.4f}  val_auc {vauc:.4f}  ({time.time()-t0:.0f}s){star}")
    model.load_state_dict(best_state); print(f'  restored best epoch {best_ep} (val_auc {best_auc:.4f})')
    return model, best_auc, best_ep
print('ready')

ready


In [5]:
# Cell 5: train all 12 (resumable), save checkpoints and float64 predictions on DDR valid and test
dev = 'cuda'; RECORD_FILE = f'{AUDIT}/ddr_training_record_v2.json'
record = json.load(open(RECORD_FILE)) if os.path.exists(RECORD_FILE) else {'started': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'models': {}}
todo = [(mi, cfg, k) for k in SEEDS for mi, cfg in enumerate(MODELS) if not os.path.exists(f"{CKPT}/ddr_{cfg['name']}{tag(k)}_v2.pt")]
print('to train this session:', [f"{c['name']}{tag(k)}" for _, c, k in todo])
for i, (mi, cfg, k) in enumerate(todo, 1):
    seed = BASE_SEED + 10 * mi + k; name = f"{cfg['name']}{tag(k)}"; eval_bs = 32 if cfg['arch'] == 'retfound' else 64
    print(f'\n=== [{i}/{len(todo)}] {name} (seed {seed}) ==='); t0 = time.time()
    model, best_auc, best_ep = train_one(cfg, dev, seed, eval_bs)
    ck = f'{CKPT}/ddr_{name}_v2.pt'
    torch.save({'state_dict': model.state_dict(), 'config': cfg, 'seed': seed, 'best_val_auc': best_auc, 'best_epoch': best_ep,
                'training_data': 'DDR official train file, locked v2 cohort'}, ck)
    for s in ('val', 'test'):
        df = splits[s]; pr = predict(model, df, dev, eval_bs)
        pd.DataFrame({'image_id': df.image_id, 'true_grade': df.level, 'true_label': df.binary_label,
                      'predicted_probability': pr, 'model_name': cfg['name'], 'replicate': k, 'training_data': 'DDR'}
                     ).to_csv(f'{PREDS}/ddrtrained_predictions_{s}_{name}_v2.csv', index=False, float_format='%.17g')
    test_auc = float(roc_auc_score(splits['test'].binary_label, pd.read_csv(f'{PREDS}/ddrtrained_predictions_test_{name}_v2.csv').predicted_probability))
    record['models'][name] = {'seed': seed, 'best_val_auc': round(best_auc, 4), 'best_epoch': best_ep, 'ddr_test_auc': round(test_auc, 4),
                              'minutes': round((time.time() - t0) / 60, 1), 'checkpoint_sha256': sha256_file(ck)}
    with open(RECORD_FILE, 'w') as f: json.dump(record, f, indent=2)
    print(f'  done in {record["models"][name]["minutes"]} min | val AUC {best_auc:.4f} | DDR test AUC {test_auc:.4f}')
    del model; torch.cuda.empty_cache()
left = [f"{c['name']}{tag(k)}" for k in SEEDS for c in MODELS if not os.path.exists(f"{CKPT}/ddr_{c['name']}{tag(k)}_v2.pt")]
print('\nremaining:', left or 'none, all 12 trained')

to train this session: ['densenet121_5class', 'efficientnet_b3_5class', 'resnet50_5class', 'retfound_5class', 'densenet121_5class_seed1', 'efficientnet_b3_5class_seed1', 'resnet50_5class_seed1', 'retfound_5class_seed1', 'densenet121_5class_seed2', 'efficientnet_b3_5class_seed2', 'resnet50_5class_seed2', 'retfound_5class_seed2']

=== [1/12] densenet121_5class (seed 20261001) ===


model.safetensors: reconstructing file:   0%|          |  0.00B / 32.3MB            

model.safetensors: downloading bytes:           |  0.00B            

  epoch 1/15  loss 0.9186  val_auc 0.9536  (181s)  <- best
  epoch 2/15  loss 0.6562  val_auc 0.9541  (25s)  <- best
  epoch 3/15  loss 0.5664  val_auc 0.9633  (25s)  <- best
  epoch 4/15  loss 0.5100  val_auc 0.9663  (25s)  <- best
  epoch 5/15  loss 0.4705  val_auc 0.9640  (25s)
  epoch 6/15  loss 0.4221  val_auc 0.9737  (25s)  <- best
  epoch 7/15  loss 0.3807  val_auc 0.9754  (25s)  <- best
  epoch 8/15  loss 0.3444  val_auc 0.9677  (25s)
  epoch 9/15  loss 0.3034  val_auc 0.9687  (25s)
  epoch 10/15  loss 0.2789  val_auc 0.9689  (25s)
  epoch 11/15  loss 0.2544  val_auc 0.9717  (25s)
  epoch 12/15  loss 0.2381  val_auc 0.9717  (25s)
  epoch 13/15  loss 0.2177  val_auc 0.9713  (25s)
  epoch 14/15  loss 0.2085  val_auc 0.9706  (25s)
  epoch 15/15  loss 0.2004  val_auc 0.9714  (25s)
  restored best epoch 7 (val_auc 0.9754)
  done in 10.0 min | val AUC 0.9754 | DDR test AUC 0.9083

=== [2/12] efficientnet_b3_5class (seed 20261011) ===


model.safetensors: reconstructing file:   0%|          |  0.00B / 49.3MB            

...

  epoch 1/15  loss 0.8528  val_auc 0.9529  (86s)  <- best
  epoch 2/15  loss 0.6702  val_auc 0.9542  (85s)  <- best
  epoch 3/15  loss 0.6180  val_auc 0.9624  (87s)  <- best
  epoch 4/15  loss 0.5744  val_auc 0.9652  (86s)  <- best
  epoch 5/15  loss 0.5424  val_auc 0.9698  (86s)  <- best
  epoch 6/15  loss 0.5209  val_auc 0.9646  (84s)
  epoch 7/15  loss 0.5000  val_auc 0.9693  (84s)
  epoch 8/15  loss 0.4807  val_auc 0.9698  (84s)
  epoch 9/15  loss 0.4682  val_auc 0.9678  (83s)
  epoch 10/15  loss 0.4441  val_auc 0.9676  (83s)
  epoch 11/15  loss 0.4406  val_auc 0.9683  (84s)
  epoch 12/15  loss 0.4365  val_auc 0.9677  (84s)
  epoch 13/15  loss 0.4260  val_auc 0.9692  (84s)
  epoch 14/15  loss 0.4068  val_auc 0.9693  (84s)
  epoch 15/15  loss 0.4007  val_auc 0.9690  (85s)
  restored best epoch 5 (val_auc 0.9698)
  done in 21.6 min | val AUC 0.9698 | DDR test AUC 0.8937

=== [5/12] densenet121_5class_seed1 (seed 20261002) ===
  epoch 1/15  loss 0.8752  val_auc 0.9536  (26s)  <- best


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

  epoch 1/15  loss 0.8666  val_auc 0.9584  (85s)  <- best
  epoch 2/15  loss 0.6753  val_auc 0.9611  (85s)  <- best
  epoch 3/15  loss 0.6104  val_auc 0.9672  (85s)  <- best
  epoch 4/15  loss 0.5670  val_auc 0.9692  (85s)  <- best
  epoch 5/15  loss 0.5442  val_auc 0.9704  (85s)  <- best
  epoch 6/15  loss 0.5165  val_auc 0.9722  (85s)  <- best
  epoch 7/15  loss 0.5023  val_auc 0.9691  (84s)
  epoch 8/15  loss 0.4769  val_auc 0.9712  (84s)
  epoch 9/15  loss 0.4612  val_auc 0.9733  (85s)  <- best
  epoch 10/15  loss 0.4551  val_auc 0.9738  (85s)  <- best
  epoch 11/15  loss 0.4368  val_auc 0.9748  (84s)  <- best
  epoch 12/15  loss 0.4289  val_auc 0.9741  (84s)
  epoch 13/15  loss 0.4091  val_auc 0.9740  (85s)
  epoch 14/15  loss 0.4166  val_auc 0.9735  (84s)
  epoch 15/15  loss 0.4021  val_auc 0.9733  (84s)
  restored best epoch 11 (val_auc 0.9748)
  done in 21.4 min | val AUC 0.9748 | DDR test AUC 0.9014

=== [9/12] densenet121_5class_seed2 (seed 20261003) ===
  epoch 1/15  loss 0.

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

  epoch 1/15  loss 0.8596  val_auc 0.9558  (85s)  <- best
  epoch 2/15  loss 0.6791  val_auc 0.9600  (85s)  <- best
  epoch 3/15  loss 0.6128  val_auc 0.9646  (84s)  <- best
  epoch 4/15  loss 0.5747  val_auc 0.9630  (84s)
  epoch 5/15  loss 0.5417  val_auc 0.9686  (84s)  <- best
  epoch 6/15  loss 0.5229  val_auc 0.9696  (85s)  <- best
  epoch 7/15  loss 0.5033  val_auc 0.9713  (85s)  <- best
  epoch 8/15  loss 0.4819  val_auc 0.9705  (84s)
  epoch 9/15  loss 0.4744  val_auc 0.9719  (85s)  <- best
  epoch 10/15  loss 0.4504  val_auc 0.9720  (85s)  <- best
  epoch 11/15  loss 0.4373  val_auc 0.9702  (84s)
  epoch 12/15  loss 0.4321  val_auc 0.9714  (84s)
  epoch 13/15  loss 0.4247  val_auc 0.9716  (84s)
  epoch 14/15  loss 0.4206  val_auc 0.9717  (84s)
  epoch 15/15  loss 0.4275  val_auc 0.9715  (84s)
  restored best epoch 10 (val_auc 0.9720)
  done in 21.4 min | val AUC 0.9720 | DDR test AUC 0.8911

remaining: none, all 12 trained


## After this notebook
When all 12 are trained, send back `99_audit/ddr_training_record_v2.json`. Next: APTOS inference for the 12 DDR models (the 33 EyePACS models already have APTOS predictions), then the pilot comparison.